# ML-06 - Signal Audit: Do the Refresh Signals Hold?

**Lane:** Refresh / Content Opportunity Scoring

This audit tests whether the assumptions behind a refresh-review baseline are visible in the 30,000-row starter dataset. The outcome used for diagnostic comparisons is `trend_direction == "down"`; it is derived from trend data, so it is never used as a feature or score input.

## 1. Distributions

I inspect distributions before interpreting a relationship. Traffic measures are heavy-tailed: a small number of pages can have vastly more impressions than the typical page. I therefore use bucket tables and medians rather than a raw Pearson correlation.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
pd.set_option("display.max_colwidth", 120)

df["is_declining_proxy"] = (df["trend_direction"] == "down").astype(int)

distribution_summary = pd.DataFrame({
    "days_since_last_update": df["days_since_last_update"].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.99, 1]),
    "impressions_90d": df["impressions_90d"].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.99, 1]),
    "log1p_impressions_90d": np.log1p(df["impressions_90d"]).quantile([0, 0.25, 0.5, 0.75, 0.9, 0.99, 1]),
}).rename_axis("quantile").round(2)

freshness_distribution = (
    df.groupby("freshness_tier", observed=True)
    .agg(n=("content_id", "size"), median_impressions_90d=("impressions_90d", "median"))
    .reset_index()
)

print(f"Rows: {len(df):,}; clients: {df['client_id'].nunique():,}; declining-proxy base rate: {df['is_declining_proxy'].mean():.3f}")
print("\nQuantiles show a heavy impression tail: use log1p or buckets for traffic-like fields.")
display(distribution_summary)

print("\nFreshness distribution (n is visible):")
display(freshness_distribution)


Rows: 30,000; clients: 32; declining-proxy base rate: 0.542

Quantiles show a heavy impression tail: use log1p or buckets for traffic-like fields.

Freshness distribution (n is visible):


,days_since_last_update,impressions_90d,log1p_impressions_90d
quantile,,,
0.00,1.0,1.00,0.69
0.25,20.0,81.00,4.41
0.50,20.0,731.00,6.60
0.75,104.0,3615.25,8.19
0.90,104.0,12136.40,9.40
0.99,106.0,73505.83,11.21
1.00,373.0,517715.00,13.16


,freshness_tier,n,median_impressions_90d
0,0-30,20480,470.0
1,181+,174,15.5
2,31-90,175,510.0
3,91-180,9171,1692.0


None

## 2. Signal tests #1 / #2 / #3

### Test 1: staleness

**Claim:** pages that have gone longer without an update are more likely to need refresh review.  
**Verdict: MIXED.** The 91-180 day group has a higher observed decline-proxy share than the 0-30 day group, but the pattern does not continue monotonically in the older group. This supports a review threshold and a capped age weight, not a claim that every additional day makes a page worse.

### Test 2: visibility

**Claim:** more search visibility identifies pages with more refresh opportunity.  
**Verdict: MIXED.** The middle impression tiers have larger observed decline-proxy shares than both low and excellent tiers. Visibility is still useful to prioritize potential impact, but it does not by itself predict decline.

### Test 3: current search position

**Claim:** pages outside the very top positions can have more room for a content intervention.  
**Verdict: CONFIRMED, directionally.** The observed decline-proxy share is highest in the `striking` position band and lower for `top_3` and `deep`. This is an association, not proof that changing content will change position.

In [2]:
# Test 1: staleness. Buckets with n < 50 are displayed but excluded from the verdict.
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"],
    include_lowest=True,
)
staleness_test = (
    df.groupby("staleness_bucket", observed=True)
    .agg(n=("content_id", "size"), median_impressions_90d=("impressions_90d", "median"), declining_proxy_share=("is_declining_proxy", "mean"))
    .reset_index()
)
staleness_test["declining_proxy_share"] = staleness_test["declining_proxy_share"].round(3)
print("Test 1 verdict: MIXED")
display(staleness_test)

# Test 2: impression tiers. These are pre-defined transparent tiers from the data dictionary.
impression_test = (
    df.groupby("impression_tier", observed=True)
    .agg(n=("content_id", "size"), median_impressions_90d=("impressions_90d", "median"), declining_proxy_share=("is_declining_proxy", "mean"))
    .reset_index()
)
impression_test["declining_proxy_share"] = impression_test["declining_proxy_share"].round(3)
print("\nTest 2 verdict: MIXED")
display(impression_test)

# Test 3: discard avg_position == 0 because the dictionary defines it as no position data.
position_data = df.loc[df["avg_position"].gt(0)].copy()
position_test = (
    position_data.groupby("position_tier", observed=True)
    .agg(n=("content_id", "size"), median_impressions_90d=("impressions_90d", "median"), declining_proxy_share=("is_declining_proxy", "mean"))
    .reset_index()
)
position_test["declining_proxy_share"] = position_test["declining_proxy_share"].round(3)
print("\nTest 3 verdict: CONFIRMED, directionally")
display(position_test)


Test 1 verdict: MIXED

Test 2 verdict: MIXED

Test 3 verdict: CONFIRMED, directionally


,staleness_bucket,n,median_impressions_90d,declining_proxy_share
0,0-30,20480,470.0,0.511
1,31-90,175,510.0,0.589
2,91-180,9171,1692.0,0.611
3,181-365,169,16.0,0.467
4,365+,5,2.0,0.600


,impression_tier,n,median_impressions_90d,declining_proxy_share
0,excellent,1078,48675.0,0.462
1,good,7205,7249.0,0.586
2,low,11248,31.0,0.454
3,moderate,10469,998.0,0.615


,position_tier,n,median_impressions_90d,declining_proxy_share
0,deep,1319,218.0,0.344
1,page_1,11814,1179.5,0.570
2,page_3_5,7242,811.5,0.562
3,striking,7304,874.5,0.610
4,top_3,1116,53.0,0.494


None

## 3. The flag-linked test

FlyRank refresh flags rely in part on **staleness**. I repeat the staleness check on the largest content-type slice, `keyword article`, to see whether the overall result is being driven by a small category.

**Verdict: MIXED.** The 91-180-day pattern remains directionally higher than the recent bucket in this large slice, while the very-old group does not establish a stronger trend. The rule should use staleness as one review signal, with a cap and human review.

In [3]:
largest_content_type = df["content_type"].value_counts().index[0]
flag_slice = df.loc[df["content_type"].eq(largest_content_type)].copy()
flag_linked_test = (
    flag_slice.groupby("staleness_bucket", observed=True)
    .agg(n=("content_id", "size"), median_impressions_90d=("impressions_90d", "median"), declining_proxy_share=("is_declining_proxy", "mean"))
    .reset_index()
)
flag_linked_test["declining_proxy_share"] = flag_linked_test["declining_proxy_share"].round(3)
print(f"Flag-linked staleness audit on content_type = {largest_content_type!r}")
print("Verdict: MIXED. n is shown for every bucket; small buckets are not used to make a strong claim.")
display(flag_linked_test)


Flag-linked staleness audit on content_type = 'keyword article'
Verdict: MIXED. n is shown for every bucket; small buckets are not used to make a strong claim.


,staleness_bucket,n,median_impressions_90d,declining_proxy_share
0,0-30,17954,698.0,0.536
1,31-90,173,510.0,0.590
2,91-180,8906,1794.0,0.612
3,181-365,169,16.0,0.467
4,365+,5,2.0,0.600


None

## 4. What this means in practice

The evidence supports a modest refresh-review rule: screen for pages that are at least 90 days since their last recorded update and have enough visibility for a review to matter. Do not interpret age or impressions as proof that a page is declining, or that a refresh will create an uplift.

Position may be a useful later refinement, but this week's baseline intentionally stays narrow and transparent. A content team should review the queue alongside context that is absent from this dataset, including seasonality, technical/indexing issues, business priority, and whether a page was changed outside the tracked workflow.

In [4]:
audit_receipt = {
    "lane": "Refresh / Content Opportunity Scoring",
    "diagnostic_outcome": "trend_direction == down (not a feature)",
    "verdicts": {
        "staleness": "MIXED",
        "visibility": "MIXED",
        "position": "CONFIRMED, directionally",
        "flag_linked_staleness_on_largest_content_type": "MIXED",
    },
    "sample_size_floor": 50,
    "careful_claim": "Associations in this snapshot support review prioritization, not causal claims.",
}
output_dir = ROOT / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
with (output_dir / "signal_audit_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(audit_receipt, handle, indent=2)
print("Wrote work/outputs/signal_audit_metrics.json")
print("The audit uses the decline proxy only as a diagnostic outcome; it is not part of the baseline score.")


Wrote work/outputs/signal_audit_metrics.json
The audit uses the decline proxy only as a diagnostic outcome; it is not part of the baseline score.


None

## Self-check

- [x] I inspected distributions and treated impressions as heavy-tailed.
- [x] Three safe signals each have a table with visible `n` values and a clear verdict.
- [x] Staleness is a real FlyRank refresh-flag signal and was checked again on a large content-type slice.
- [x] Buckets with fewer than 50 rows are not used for a strong conclusion.
- [x] The trend-derived decline proxy is a diagnostic outcome only, never a feature.
- [x] Claims are observational, directional, and appropriate for decision support.